## 11. Hooks：把 agent 变成可控系统

> 来源：[Intercept and control agent behavior with hooks](https://code.claude.com/docs/en/agent-sdk/hooks)

Hook **不是给 Claude 用的工具**，而是给"包在 agent 循环外面的应用控制层"用的回调：在生命周期的确定时机跑你的代码，用来拦截危险操作、审计合规、改写输入输出、脱敏、注入 context。Hooks 跑在你的应用进程里、不占 context window；`PreToolUse` 的 deny 是唯一连 `bypassPermissions` 都拦得住的机制（见 §9.1「权限评估顺序」）。


### 11.1 Python 可用的 hook 事件（10 个）

| 事件 | 触发时机 | 典型用途 |
|---|---|---|
| `PreToolUse` | tool 执行前（可阻断/改写） | 拦危险命令、重定向路径 |
| `PostToolUse` | tool 返回后 | 审计日志；`additionalContext` 往 tool result 后追加信息（旧字段 `updatedMCPToolOutput` 只能替换 MCP 输出，已废弃） |
| `PostToolUseFailure` | tool 执行失败 | 记录/处理 tool 错误 |
| `UserPromptSubmit` | prompt 提交时 | 注入额外 context（`additionalContext`）；**不要在这里 spawn 子 agent**——子 agent 的 prompt 又触发本 hook，递归死循环 |
| `Stop` | agent 执行停止 | 保存状态（忽略 matcher） |
| `SubagentStart` / `SubagentStop` | 子 agent 启动/完成 | 跟踪并行任务；`SubagentStop` 输入含 `agent_transcript_path`（子 agent transcript 路径）、`stop_hook_active` |
| `PreCompact` | compaction 前 | 归档完整 transcript |
| `PermissionRequest` | 将弹权限询问时 | 发外部通知（Slack/邮件） |
| `Notification` | agent 状态消息 | 转发状态到监控系统；子类型有 `permission_prompt` / `idle_prompt` / `auth_success` / `elicitation_*`，文本在 `message`（可选 `title`）字段 |

两点补充：`agent_id` / `agent_type` 这两个输入字段只在 `PreToolUse` / `PostToolUse` / `PostToolUseFailure` 三个事件里有，用于区分回调发生在主 agent 还是哪个子 agent；`SessionStart`/`SessionEnd`/`PostToolBatch`/`MessageDisplay` 等仅 TypeScript 有，Python 想跑 session 级 hook 只能用 settings.json 里的 shell command hook，经 `setting_sources=["project"]` 加载。

### 11.2 配置结构与 matcher 规则

```python
hooks={"PreToolUse": [HookMatcher(matcher="Write|Edit", hooks=[callback], timeout=60)]}
```

matcher 匹配规则（大小写敏感，只匹配 tool 名，不匹配文件路径等参数）：

- 只含字母数字、`_`、`-`、空格、`,`、`|` → **精确匹配**，`|` 或 `,` 分隔多候选（`"Write|Edit"`）。注意：`-` 进精确匹配字符集需要 Claude Code ≥ v2.1.195，更早版本把 `code-reviewer` 这类带连字符的名字按未锚定正则求值。
- `*`、空串、省略 → 匹配该事件全部发生。
- 含其他字符 → 按**未锚定正则**求值（`"^mcp__"` 匹配所有 MCP tool；`"Edit.*"` 同时命中 `Edit` 和 `NotebookEdit`）。

> [!warning] MCP matcher 高频坑
> `matcher="mcp__memory"` 落在精确匹配字符集里，只会精确比较、**匹配不到任何 tool**。要匹配某 server 全部工具写 `mcp__memory__.*`。

### 11.3 回调签名与返回值

```python
async def my_hook(input_data: dict, tool_use_id: str | None, context) -> dict: ...
# context 参数在 Python 中是保留位，当前没有内容
```

`input_data` 随事件而异（`PreToolUse` 有 `tool_name`/`tool_input`；全部事件共享 `session_id`/`cwd`/`hook_event_name`）。返回 `{}` = 放行。要干预就返回：

```python
{
    "systemMessage": "给用户看的提示（可选）",
    "continue_": True,          # 顶层字段；Python 用下划线避开关键字
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "allow" | "deny" | "ask" | "defer",
        "permissionDecisionReason": "给模型看的原因",
        "updatedInput": {...},  # 改写入参；必须配 allow 或 ask 才生效（defer 时被忽略）
    },
}
```

- **`defer` 的语义**：结束本次 query、留待之后 resume 再继续——适合"等人审批可能超过进程存活时间"的场景（§9.3「can_use_tool 回调」 提到的出路就是它）。
- 多个 hook 命中同一事件时**并行执行**，权限决策取最严：`deny > defer > ask > allow`。
- 只做副作用（日志/webhook）不想拖慢 agent 时，返回 `{"async_": True, "asyncTimeout": 30000}` 让 agent 立即继续（`asyncTimeout` 毫秒、可选；async 输出不能再阻断或改写）。
- **`systemMessage` 默认不出现在消息流**——要在应用侧看到 hook 的输出，得开 `include_hook_events=True`（对应 §7.2「SystemMessage 家族」 的 `HookEventMessage`）。

**Hook vs `can_use_tool` 怎么选**：审批交互（等真人点头）用 `can_use_tool`；必须每次调用都生效的确定性检查、审计、多生命周期点干预用 Hooks。

> [!note] 排障速查
> hook 不触发：事件名大小写、matcher 精确度、`max_turns` 触限时 hook 可能来不及跑。改写不生效：`updatedInput` 必须在 `hookSpecificOutput` 内且带 `permissionDecision: "allow"`。hook 内抛异常会打断 agent——发 HTTP 等副作用要自己 try/except，阻塞调用用 `asyncio.to_thread` 包。子 agent 里同样的权限提示成倍弹出：子 agent 不继承父级会话的临时批准，用 PreToolUse 的 allow 决策或权限规则统一放行（§14.2「信息边界」）。

In [ ]:
from datetime import datetime
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher, ResultMessage


# PreToolUse：阻断对 .env 的任何写入，reason 给模型、systemMessage 给用户
async def protect_env_files(input_data, tool_use_id, context):
    file_path = input_data["tool_input"].get("file_path", "")
    if file_path.split("/")[-1] == ".env":
        return {
            "systemMessage": ".env is protected.",
            "hookSpecificOutput": {
                "hookEventName": input_data["hook_event_name"],
                "permissionDecision": "deny",
                "permissionDecisionReason": "Cannot modify .env files",
            },
        }
    return {}


# PostToolUse：把所有文件改动写进审计日志
async def log_file_change(input_data, tool_use_id, context):
    fp = input_data.get("tool_input", {}).get("file_path", "unknown")
    with open("./audit.log", "a") as f:
        f.write(f"{datetime.now()}: modified {fp}\n")
    return {}  # 返回空 dict = 放行


async def demo_hook():
    async for message in query(
        prompt="Refactor utils.py to improve readability",
        options=ClaudeAgentOptions(
            permission_mode="acceptEdits",
            hooks={
                "PreToolUse": [
                    HookMatcher(matcher="Write|Edit", hooks=[protect_env_files])
                ],
                "PostToolUse": [
                    HookMatcher(matcher="Edit|Write", hooks=[log_file_change])
                ],
            },
        ),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_hook()